In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
df = pd.read_csv('heart_disease.csv')

target_col = 'Heart Disease Status'
df = df.dropna(subset=[target_col]).copy()

y_raw = df[target_col]
y_num = pd.to_numeric(y_raw, errors='coerce')
if y_num.notna().all():
    y_all = y_num.astype(int)
else:
    y_all = y_raw.astype(str).str.lower().str.strip().map({'yes': 1, 'no': 0, '1': 1, '0': 0}).fillna(0).astype(int)

# Make the dataset 50/50 by undersampling majority class.
df_pos = df[y_all == 1].copy()
df_neg = df[y_all == 0].copy()
n_min = min(len(df_pos), len(df_neg))

balanced_df = pd.concat(
    [
        df_pos.sample(n=n_min, random_state=42),
        df_neg.sample(n=n_min, random_state=42),
    ],
    axis=0,
).sample(frac=1, random_state=42).reset_index(drop=True)

# Features requested from your screenshot (classic heart feature names)
requested_from_image = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

# Map screenshot names to this dataset's column names where possible.
feature_name_map = {
    'age': 'Age',
    'sex': 'Gender',
    'trestbps': 'Blood Pressure',
    'chol': 'Cholesterol Level',
    'fbs': 'Fasting Blood Sugar',
}

mapped_candidates = [feature_name_map.get(f, f) for f in requested_from_image]
feature_cols = [c for c in mapped_candidates if c in balanced_df.columns and c != target_col]
missing_from_dataset = [c for c in mapped_candidates if c not in balanced_df.columns]

if not feature_cols:
    raise ValueError('None of the requested screenshot features exist in this dataset after mapping.')

# De-duplicate while preserving order.
feature_cols = list(dict.fromkeys(feature_cols))

X = balanced_df[feature_cols].copy()
y = balanced_df[target_col].astype(str).str.lower().str.strip().map({'yes': 1, 'no': 0, '1': 1, '0': 0}).fillna(0).astype(int)

print('Rows used after 50/50 balancing:', len(balanced_df))
print('Target balance (50/50):')
print(y.value_counts())
print('Requested (image) features:', requested_from_image)
print('Mapped and used features:', feature_cols)
print('Mapped but missing in dataset:', missing_from_dataset)

Rows used after 50/50 balancing: 4000
Target balance (50/50):
Heart Disease Status
1    2000
0    2000
Name: count, dtype: int64
Requested (image) features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Mapped and used features: ['Age', 'Gender', 'Blood Pressure', 'Cholesterol Level', 'Fasting Blood Sugar']
Mapped but missing in dataset: ['cp', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [3]:

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, test_idx = next(sss.split(X, y))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Multi-feature model (using only mapped screenshot features that exist)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4, zero_division=0))
print('ROC AUC  :', round(roc_auc_score(y_test, y_prob), 4))

Multi-feature model (using only mapped screenshot features that exist)

Classification Report:
              precision    recall  f1-score   support

           0     0.4878    0.5000    0.4938       200
           1     0.4872    0.4750    0.4810       200

    accuracy                         0.4875       400
   macro avg     0.4875    0.4875    0.4874       400
weighted avg     0.4875    0.4875    0.4874       400

ROC AUC  : 0.5031
